In [1]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [2]:
def get_preprocessor(numeric_features, categorical_features):
    """Constrói a pipeline de transformação de dados (imputação e escala)."""
    num_transf = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                                 ('scaler', StandardScaler())])
    cat_transf = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                                 ('onehot', OneHotEncoder(handle_unknown='ignore'))])
    
    return ColumnTransformer(transformers=[('num', num_transf, numeric_features),
                                           ('cat', cat_transf, categorical_features)])

def train_rf_model(X_train, y_train, preprocessor):
    """Treina o modelo Random Forest focado em Precisão (evitando Data Leakage e Overfitting)."""
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
    ])
    
    param_grid = {'classifier__n_estimators': [100, 200], 'classifier__max_depth': [6, 10]}
    cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    grid = GridSearchCV(clf, param_grid, scoring='precision', cv=cv_strategy, n_jobs=-1)
    grid.fit(X_train, y_train)
    return grid.best_estimator_